# Perfil territorial x pobreza x educacion

Cruce multi-capa por `codigo_comuna`: la tesis de chile-hub en un solo notebook.

**Capas utilizadas:**
- `perfil_territorial_comunal` — tabla derivada con 42 metricas por comuna (DPA + Censo + salud + educacion + finanzas + SIEDU)
- `pobreza_comunal` — tasas de pobreza por ingresos y multidimensional (SAE/MDS)
- `resultados_educacionales` — metricas educacionales agregadas por comuna y anio (MINEDUC)

**Invariante del hub:** `codigo_comuna` siempre es `str` de 5 caracteres con cero inicial (p. ej. `"01101"` para Iquique).
Esta convencion garantiza joins sin perdida de datos entre capas.

In [ ]:
import polars as pl

from chile_hub import ChileHub

hub = ChileHub()

perfil = hub.load_polars("perfil_territorial_comunal")
pobreza = hub.load_polars("pobreza_comunal")
educacion = hub.load_polars("resultados_educacionales")

print(f"perfil_territorial_comunal: {perfil.shape}")
print(f"pobreza_comunal:            {pobreza.shape}")
print(f"resultados_educacionales:   {educacion.shape}")

## Cruce perfil x pobreza por ingresos

Filtramos `pobreza_comunal` a la dimension `ingresos` (la mas comparable comunalmente).
El join es posible porque ambas capas usan el mismo `codigo_comuna` de 5 chars — sin limpieza adicional.

In [ ]:
# Filtrar pobreza por ingresos y unir con el perfil
pobreza_ingresos = pobreza.filter(pl.col("dimension") == "ingresos")

perfil_pobreza = perfil.join(
    pobreza_ingresos.select("codigo_comuna", "tasa", "limite_inferior", "limite_superior"),
    on="codigo_comuna",
    how="inner",
).rename({"tasa": "tasa_pobreza_ingresos"})

print(f"Comunas con datos de pobreza por ingresos: {len(perfil_pobreza)}")
print("\nCobertura disponible (nota: pobreza_comunal usa muestra SAE — cobertura parcial):")
print(
    perfil_pobreza.select(
        "nombre_comuna",
        "nombre_region",
        "poblacion_censada",
        "tasa_pobreza_ingresos",
        "establecimientos_educacionales_total",
    )
)

## Agregar resultados educacionales (ano mas reciente)

`resultados_educacionales` puede tener multiples anos; tomamos el mas reciente disponible
para cada comuna sin hardcodear el ano.

In [ ]:
# Tomar el anio mas reciente disponible para cada comuna
anio_max = educacion.select(pl.col("anio").max()).item()
edu_reciente = educacion.filter(pl.col("anio") == anio_max)

print(f"Anio mas reciente en resultados_educacionales: {anio_max}")
print(f"Comunas con datos educacionales: {len(edu_reciente)}")

# Cruce triple: perfil x pobreza x educacion
analisis = perfil_pobreza.join(
    edu_reciente.select(
        "codigo_comuna",
        "tasa_aprobacion",
        "asistencia_promedio",
        "matricula_total",
        "establecimientos_reportados",
    ),
    on="codigo_comuna",
    how="left",
)

print(f"\nComunas en el cruce triple: {len(analisis)}")

## Ranking y correlacion

Ordenamos por tasa de pobreza y calculamos la correlacion de Pearson entre
pobreza por ingresos y tasa de aprobacion escolar.

In [ ]:
# Ranking por tasa de pobreza (mayor a menor)
ranking = analisis.select(
    "nombre_comuna",
    "nombre_region",
    "poblacion_censada",
    "tasa_pobreza_ingresos",
    "tasa_aprobacion",
    "asistencia_promedio",
).sort("tasa_pobreza_ingresos", descending=True)

print("Comunas por tasa de pobreza por ingresos (mayor a menor):")
print(ranking)

# Correlacion de Pearson entre pobreza y aprobacion escolar
corr = analisis.select(
    pl.corr("tasa_pobreza_ingresos", "tasa_aprobacion").alias("pearson_pobreza_vs_aprobacion")
).item()

print(f"\nCorrelacion Pearson (pobreza por ingresos vs tasa de aprobacion): {corr:.3f}")
print("(valor negativo indica que mayor pobreza se asocia a menor aprobacion)")

## Lecturas del analisis

- El join triple funciono sin ninguna limpieza de datos: `codigo_comuna` como llave estable de 5 chars es suficiente.
- La cobertura de `pobreza_comunal` es parcial (muestra SAE/MDS — no todas las 346 comunas); los cruces con `left` join preservan la informacion del perfil para todas las comunas.
- `resultados_educacionales` cubre 345 de 346 comunas para el anio disponible.
- La correlacion negativa esperada entre pobreza e indicadores educacionales puede validarse ampliando la muestra cuando el MDS publique datos SAE completos.

## Como extender este analisis

- Reemplazar `pobreza_comunal` por `delincuencia_comunal` (carril `candidate`) para explorar seguridad vs. educacion.
- Cruzar con `consumo_electrico_comunal` para agregar una dimension economica.
- Usar `nombre_comuna_clean` (minusculas sin tildes) para unir datos propios que no traigan `codigo_comuna`.
- Agregar `indicadores_urbanos_siedu` para la dimension de infraestructura urbana.

## Referencias

- [docs/datasets/perfil_territorial_comunal.md](https://github.com/cortega26/chile-hub/blob/main/docs/datasets/perfil_territorial_comunal.md)
- [docs/datasets/pobreza_comunal.md](https://github.com/cortega26/chile-hub/blob/main/docs/datasets/pobreza_comunal.md)
- [README del repositorio](https://github.com/cortega26/chile-hub)